In this tutorial, we are going to evaluate the performance of the naive RAG and the GraphRAG algorithm on a [multi-hop RAG task](https://github.com/yixuantt/MultiHop-RAG).

## Setup
Make sure you install the necessary dependencies by running the following commands:

Import the necessary libraries, and set up your openai api key if needed:

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
#os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
import json
import sys
sys.path.append("../..")

import nest_asyncio
nest_asyncio.apply()
import logging

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.INFO)
from nano_graphrag import GraphRAG, QueryParam
from datasets import Dataset 
from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    answer_similarity,
)

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Download the dataset from [Github Repo](https://github.com/yixuantt/MultiHop-RAG/tree/main/dataset). 
If should contain two files:
- `MultiHopRAG.json`
- `corpus.json`

After downloading the dataset, replace the below paths to the paths on your machine.

In [3]:

multi_hop_rag_file = "./fixtures/MultiHopRAG.json"
multi_hop_corpus_file = "./fixtures/corpus.json"

## Preprocess

In [4]:

with open(multi_hop_rag_file) as f:
    multi_hop_rag_dataset = json.load(f)
with open(multi_hop_corpus_file) as f:
    multi_hop_corpus = json.load(f)

corups_url_refernces = {}
for cor in multi_hop_corpus:
    corups_url_refernces[cor['url']] = cor

We only use the top-100 queries for evaluation.

In [5]:
import pandas as pd
# Load toy datasets
documents_df = pd.read_csv("./fixtures/toydataset/documents.csv")
queries_df = pd.read_csv("./fixtures/toydataset/multi_passage_answer_questions.csv")

# Prepare corpus
total_corpus = documents_df['text'].tolist()

Add index for the `total_corups` using naive RAG and GraphRAG

In [14]:
'''
# if you want to clear existing cache
import shutil
import os

cache_dir = "nano_graphrag_cache_toy_dataset_rag_test"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
'''


'\n'

In [17]:
import ollama
import numpy as np
from nano_graphrag._utils import compute_args_hash, wrap_embedding_func_with_attrs

# Ollama settings
MODEL = "deepseek-r1"  # or any other model you have in Ollama
EMBEDDING_MODEL = "deepseek-r1"
EMBEDDING_MODEL_DIM = 3584
EMBEDDING_MODEL_MAX_TOKENS = 8192

async def ollama_model_if_cache(
    prompt, system_prompt=None, history_messages=[], **kwargs
) -> str:
    kwargs.pop("max_tokens", None)
    kwargs.pop("response_format", None)

    ollama_client = ollama.AsyncClient()
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    hashing_kv = kwargs.pop("hashing_kv", None)
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    if hashing_kv is not None:
        args_hash = compute_args_hash(MODEL, messages)
        if_cache_return = await hashing_kv.get_by_id(args_hash)
        if if_cache_return is not None:
            return if_cache_return["return"]
            
    response = await ollama_client.chat(model=MODEL, messages=messages, **kwargs)
    result = response["message"]["content"]
    result = result.split("</think>")[-1] # 去掉<think>
    if hashing_kv is not None:
        await hashing_kv.upsert({args_hash: {"return": result, "model": MODEL}})
    return result

@wrap_embedding_func_with_attrs(
    embedding_dim=EMBEDDING_MODEL_DIM,
    max_token_size=EMBEDDING_MODEL_MAX_TOKENS,
)
async def ollama_embedding(texts: list[str]) -> np.ndarray:
    embed_text = []
    for text in texts:
        data = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
        embedding = np.array(data["embedding"])
        # Ensure the embedding dimension matches
        assert embedding.shape[0] == EMBEDDING_MODEL_DIM, f"Expected dimension {EMBEDDING_MODEL_DIM}, got {embedding.shape[0]}"
        embed_text.append(embedding)
    return np.vstack(embed_text)



In [16]:
# First time indexing will cost many time, roughly 15~20 minutes
from nano_graphrag._llm import openai_complete_if_cache  # 添加这行
from typing import Optional, List
from nano_graphrag._utils import compute_args_hash

graphrag_func = GraphRAG(
    working_dir="nano_graphrag_cache_toy_dataset_rag_test",
    enable_naive_rag=True,
    embedding_func_max_async=4,
    embedding_batch_num=64,
    best_model_func=ollama_model_if_cache,
    cheap_model_func=ollama_model_if_cache,
    embedding_func=ollama_embedding
)

graphrag_func.insert(total_corpus)

INFO:nano-graphrag:Creating working directory nano_graphrag_cache_toy_dataset_rag_test
INFO:nano-graphrag:Load KV full_docs with 0 data
INFO:nano-graphrag:Load KV text_chunks with 0 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 0 data
INFO:nano-graphrag:[New Docs] inserting 20 docs
INFO:nano-graphrag:[New Chunks] inserting 155 chunks
INFO:nano-graphrag:Insert chunks for naive RAG
INFO:nano-graphrag:Inserting 155 vectors to chunks
INFO:nano-graphrag:[Entity Extraction]...


Look at the response of different RAG methods on the first query:

In [24]:
response_formate = "Single phrase or sentence, concise and no redundant explanation needed. If you don't have the answer in context, Just response 'Insufficient information'"
naive_rag_query_param = QueryParam(mode='naive', response_type=response_formate)
naive_rag_query_only_context_param = QueryParam(mode='naive', only_need_context=True)
local_graphrag_query_param = QueryParam(mode='local', response_type=response_formate)
local_graphrag_only_context__param = QueryParam(mode='local', only_need_context=True)

In [8]:
query = multi_hop_rag_dataset[0]
print("Question:", query['query'])
print("GroundTruth Answer:", query['answer'])

Question: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?
GroundTruth Answer: Sam Bankman-Fried


In [9]:
print("NaiveRAG Answer:", graphrag_func.query(query['query'], param=naive_rag_query_param))

INFO:nano-graphrag:Truncate 20 to 12 chunks


NaiveRAG Answer: Sam Bankman-Fried


In [10]:
print("Local GraphRAG Answer:", graphrag_func.query(query['query'], param=local_graphrag_query_param))

INFO:nano-graphrag:Using 20 entites, 3 communities, 124 relations, 3 text units


Local GraphRAG Answer: Sam Bankman-Fried


Great! Now we're ready to evaluate more detailed metrics. We will use [ragas](https://docs.ragas.io/en/stable/) to evalue the answers' quality.

In [11]:
questions = [q['query'] for q in multi_hop_rag_dataset]
labels = [q['answer'] for q in multi_hop_rag_dataset]

In [12]:
from tqdm import tqdm
logging.getLogger("nano-graphrag").setLevel(logging.WARNING)

naive_rag_answers = [
    graphrag_func.query(q, param=naive_rag_query_param) for q in tqdm(questions)
]

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [03:53<00:00,  2.33s/it]


In [14]:
local_graphrag_answers = [
    graphrag_func.query(q, param=local_graphrag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [09:10<00:00,  5.50s/it]


In [34]:
naive_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": naive_rag_answers,
    }),
    metrics=[
        # answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
)

Evaluating: 100%|██████████| 200/200 [00:32<00:00,  6.19it/s]


In [36]:
local_graphrag_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": local_graphrag_answers,
    }),
    metrics=[
        # answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
)

Evaluating: 100%|██████████| 200/200 [00:23<00:00,  8.59it/s]


In [39]:
print("Naive RAG results", naive_results)
print("Local GraphRAG results", local_graphrag_results)

Naive RAG results {'answer_correctness': 0.5896, 'answer_similarity': 0.8935}
Local GraphRAG results {'answer_correctness': 0.7380, 'answer_similarity': 0.8619}
